In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Setup and Model Initialization

In [8]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="qwen3:14b", temperature=0)

In [3]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "travel_server": {
            "transport": "streamable_http",
            "url": "https://mcp.kiwi.com"
        }
    }
)

tools = await client.get_tools()

In [4]:
from typing import Dict, Any
from tavily import TavilyClient
from langchain.tools import tool

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:
    """Search the web for information"""
    return tavily_client.search(query)

In [5]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")

@tool
def query_playlist_db(query: str) -> str:
    """Query the database for playlist information"""
    try:
        return db.run(query)
    except Exception as e:
        return f"Error querying database: {e}"

## Create State

In [6]:
# FIXED STATE SCHEMA - includes required LangGraph fields
from typing import TypedDict, List, Annotated, Optional
from langchain_core.messages import BaseMessage
import operator

class WeddingState(TypedDict):
    """State schema for wedding coordinator - includes ALL required LangGraph keys"""
    # Required by LangGraph agent executor:
    messages: Annotated[List[BaseMessage], operator.add]  # Message accumulation
    remaining_steps: int  # Execution step counter (prevents infinite loops)
    
    # Custom wedding fields:
    origin: Optional[str]
    destination: Optional[str]
    guest_count: Optional[str]
    genre: Optional[str]
    wedding_date: Optional[str]

## Create Subagents

In [7]:
# FIXED: Use create_agent from langchain.agents (not deprecated prebuilt)
from langchain.agents import create_agent
from datetime import datetime

current_date = datetime.now().strftime("%Y-%m-%d")

# Travel agent - uses MCP tools
travel_agent = create_agent(
    model=model,
    tools=tools,
    state_schema=WeddingState,
    prompt=f"""
    You are a travel agent. Today's date is {current_date}.
    
    Your task is to search for flights for a wedding on the SPECIFIC DATE provided.
    
    CRITICAL RULES:
    1. The 'departureDate' MUST be exactly the date provided in the request.
    2. If no date is provided, choose a reasonable date at least 2 months in the future.
    3. The date MUST be in YYYY-MM-DD format.
    4. NEVER search for a date before {current_date}.
    
    Return a concise summary of flight options with prices and times.
    """
)

/tmp/ipykernel_114697/626132913.py:7: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  travel_agent = create_react_agent(


ValueError: Missing required key(s) {'remaining_steps'} in state_schema

In [ ]:
# Venue agent - uses web search
venue_agent = create_agent(
    model=model,
    tools=[web_search],
    state_schema=WeddingState,
    prompt="""
    You are a venue specialist. Search for venues in the desired location with the desired capacity.
    
    You must find the best venue options based on:
    - Price (lowest first)
    - Capacity (must accommodate exact guest count)
    - Reviews (highest rated)
    
    Make multiple searches if needed. Return a structured summary with:
    - Venue name, Location, Capacity, Price estimate, Review highlights
    """
)

In [ ]:
# Playlist agent - uses SQL database
playlist_agent = create_agent(
    model=model,
    tools=[query_playlist_db],
    state_schema=WeddingState,
    prompt="""
    You are a playlist specialist. Query the SQL database to curate a wedding playlist for the given genre.
    
    Steps:
    1. Query the database for tracks matching the genre
    2. Calculate total duration and total cost (each song has a price)
    3. If queries fail, adjust and retry - do not give up
    4. Return: song titles, artists, total duration (minutes), total cost, brief rationale
    """
)

## Create Tools

In [ ]:
from langchain_core.tools import tool
from langchain_core.messages import ToolMessage, HumanMessage
from langgraph.types import Command

@tool
async def search_flights(state: WeddingState) -> str:
    """Travel agent searches for flights using state values."""
    origin = state.get("origin", "") or ""
    destination = state.get("destination", "") or ""
    wedding_date = state.get("wedding_date", "") or ""
    
    if not all([origin, destination, wedding_date]):
        missing = [k for k, v in {"origin": origin, "destination": destination, "wedding_date": wedding_date}.items() if not v.strip()]
        return f"Error: Missing required fields: {', '.join(missing)}. Call update_state first."
    
    query = f"Find flights from {origin} to {destination} departing on {wedding_date} for a wedding. Return concise options with prices."
    
    # Pass state fields (excluding messages) to agent
    agent_input = {k: v for k, v in state.items() if k not in ['messages', 'remaining_steps']}
    agent_input["messages"] = [HumanMessage(content=query)]
    
    response = await travel_agent.ainvoke(agent_input)
    
    # Safely extract last message content
    messages = response.get("messages", [])
    if messages and hasattr(messages[-1], "content"):
        return messages[-1].content
    return "No flight results found."

In [ ]:
@tool
def search_venues(state: WeddingState) -> str:
    """Venue agent finds the best venue for location and capacity."""
    destination = state.get("destination", "") or ""
    capacity = state.get("guest_count", "") or ""
    
    if not destination or not capacity:
        missing = [k for k, v in {"destination": destination, "guest_count": capacity}.items() if not v.strip()]
        return f"Error: Missing: {', '.join(missing)}. Call update_state first."
    
    query = f"Find wedding venues in {destination} for {capacity} guests. Prioritize low price, exact capacity, high reviews."
    
    agent_input = {k: v for k, v in state.items() if k not in ['messages', 'remaining_steps']}
    agent_input["messages"] = [HumanMessage(content=query)]
    
    response = venue_agent.invoke(agent_input)
    
    messages = response.get("messages", [])
    if messages and hasattr(messages[-1], "content"):
        return messages[-1].content
    return "No venue results found."

In [ ]:
@tool
def suggest_playlist(state: WeddingState) -> str:
    """Playlist agent curates a wedding playlist for the given genre."""
    genre = state.get("genre", "").strip()
    
    if not genre:
        return "Error: Missing 'genre' in state. Call update_state first."
    
    query = f"Find {genre} tracks suitable for a wedding playlist from the database. Include song name, artist, duration, and price. Calculate total duration and cost."
    
    response = playlist_agent.invoke({"messages": [HumanMessage(content=query)], **{k: v for k, v in state.items() if k != 'messages'}})
    
    messages = response.get("messages", [])
    if messages:
        last_msg = messages[-1]
        return last_msg.content if hasattr(last_msg, 'content') else str(last_msg)
    return "No playlist results found."

In [ ]:
@tool
def update_state(origin: str, destination: str, guest_count: str, genre: str, wedding_date: str) -> Command:
    """Update the WeddingState with all wedding details."""
    return Command(update={
        "origin": origin.strip(),
        "destination": destination.strip(), 
        "guest_count": guest_count.strip(),
        "genre": genre.strip(),
        "wedding_date": wedding_date.strip()
    })

## Main Coordinator


In [ ]:
from datetime import datetime
from langgraph.prebuilt import create_react_agent

current_date_formatted = datetime.now().strftime("%A, %B %d, %Y")

coordinator = create_react_agent(
    model=model,
    tools=[search_flights, search_venues, suggest_playlist, update_state],
    state_schema=WeddingState,
    prompt=f"""
    You are a wedding coordinator. Today is {current_date_formatted}.
    
    TASK FLOW:
    1. Extract from user request: origin, destination, guest_count, genre, and wedding timing.
    2. Determine wedding_date:
       - If user provides specific date → use it (convert to YYYY-MM-DD)
       - If user says "in X months" → calculate from {current_date_formatted}
       - If no date mentioned → default to exactly 6 months from today
    3. IMMEDIATELY call update_state with ALL five values (origin, destination, guest_count, genre, wedding_date in YYYY-MM-DD).
    4. AFTER state is updated, call specialists IN PARALLEL if possible:
       - search_flights → get flight options
       - search_venues → get venue options  
       - suggest_playlist → get playlist with duration/cost
    5. Synthesize ALL outputs into ONE comprehensive wedding plan summary.
    6. Format the final response clearly with sections: ✈️ Flights, 🏛️ Venue, 🎵 Playlist, 📅 Date, 💰 Estimated Costs.
    
    Never proceed to specialists before calling update_state. Always wait for all specialist results before summarizing.
    """
)

## Test

In [ ]:
from langchain_core.messages import HumanMessage

# Initial state with empty defaults to prevent KeyErrors
initial_input = {
    "messages": [
        HumanMessage(content="I'm from London and I'd like a wedding in Paris for 100 guests, jazz-genre, in about three months")
    ],
    "origin": "",
    "destination": "", 
    "guest_count": "",
    "genre": "",
    "wedding_date": ""
}

# Execute the coordinator agent
response = await coordinator.ainvoke(initial_input)

# Print the final consolidated wedding plan
print("\n" + "="*50)
print("🎊 COMPLETE WEDDING PLAN 🎊")
print("="*50)
print(response["messages"][-1].content)
print("="*50)

link to trace: https://smith.langchain.com/public/7b5fe668-d3e3-4af4-b513-a8cacc0c9e84/r